In [ ]:
import numpy as np
import pandas as pd
import os
import librosa

In [ ]:
audio_path = r"C:\Users\w10\Downloads\archive (2)\audio_speech_actors_01-24"

files = []

for actor_folder in os.listdir(audio_path):
    
    actor_path = os.path.join(audio_path,actor_folder)
    
    if os.path.isdir(actor_path):
        
        for file in os.listdir(actor_path):
            
            if file.endswith(".wav"):
                files.append(os.path.join(actor_path,file))

print("Total files:",len(files))

print("\nFirst 5 files:")
for f in files[:5]:
    print(f)

Total files: 1440

First 5 files:
C:\Users\w10\Downloads\archive (2)\audio_speech_actors_01-24\Actor_01\03-01-01-01-01-01-01.wav
C:\Users\w10\Downloads\archive (2)\audio_speech_actors_01-24\Actor_01\03-01-01-01-01-02-01.wav
C:\Users\w10\Downloads\archive (2)\audio_speech_actors_01-24\Actor_01\03-01-01-01-02-01-01.wav
C:\Users\w10\Downloads\archive (2)\audio_speech_actors_01-24\Actor_01\03-01-01-01-02-02-01.wav
C:\Users\w10\Downloads\archive (2)\audio_speech_actors_01-24\Actor_01\03-01-02-01-01-01-01.wav


In [ ]:
max_len = 0
lengths = []

for actor_folder in os.listdir(audio_path):

    actor_path = os.path.join(audio_path,actor_folder)

    if os.path.isdir(actor_path):

        for file in os.listdir(actor_path):

            if file.endswith(".wav"):

                path = os.path.join(actor_path,file)

                # Wav2Vec2 uses 16kHz
                y, sr = librosa.load(path,sr=16000)

                lengths.append(len(y))

                max_len=max(max_len,len(y))

c:\Users\w10\.pyenv\pyenv-win\versions\3.10.11\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
print(max_len)
max_len = int(np.percentile(lengths,90))

66200


In [ ]:
X = []
y_label = []

def add_noise(data):
    noise = np.random.randn(len(data))
    return data + 0.003 * noise

def shift(data):
    shift_range = int(np.random.uniform(-0.1,0.1) * len(data))
    
    return np.roll(data,shift_range)

for actor_folder in os.listdir(audio_path):

    actor_path = os.path.join(audio_path,actor_folder)

    if os.path.isdir(actor_path):

        for file in os.listdir(actor_path):

            if file.endswith(".wav"):

                path = os.path.join(actor_path,file)

                y, sr = librosa.load(path,sr=16000)

                emotion = file.split("-")[2]

                signals = [y,add_noise(y),shift(y)]

                for signal in signals:

                    signal = (signal - np.mean(signal)) / (np.std(signal) + 1e-8)

                    if len(signal) < max_len:

                        pad = max_len - len(signal)

                        signal = np.pad(signal,(0,pad),mode='constant')

                    else:
                        signal = signal[:max_len]

                    X.append(signal.astype(np.float32))

                    y_label.append(emotion)

X = np.array(X,dtype=np.float32)

y_label = np.array(y_label)

print("Shape:",X.shape)

Shape: (4320, 66200)


In [81]:
X=np.array(X)
y_label=np.array(y_label)

X[0][0]

np.float32(-0.00019370086)

In [82]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y_label = encoder.fit_transform(y_label)

In [83]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,y_label,test_size=0.2,random_state=42
)

In [84]:
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

mean = np.mean(X_train,axis=0)
std = np.std(X_train,axis=0)

X_train = (X_train - mean) / (std + 1e-8)

X_test = (X_test - mean) / (std + 1e-8)

In [85]:
print(X_train.shape)
print(X_test.shape)

(3456, 66200)
(864, 66200)


In [ ]:
import tensorflow as tf
import tensorflow_hub as hub

yamnet = hub.load("https://tfhub.dev/google/yamnet/1")

X_train_embed=[]

for x in X_train:

    waveform=tf.convert_to_tensor(x,dtype=tf.float32)

    scores, embeddings, spectrogram = yamnet(waveform)

    embedding=tf.reduce_mean(embeddings,axis=0)

    X_train_embed.append(embedding.numpy())

X_train_embed=np.array(X_train_embed)

X_test_embed=[]

for x in X_test:

    waveform=tf.convert_to_tensor(x,dtype=tf.float32)

    scores, embeddings, spectrogram=yamnet(waveform)

    embedding=tf.reduce_mean(embeddings,axis=0)

    X_test_embed.append(embedding.numpy())

X_test_embed=np.array(X_test_embed)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.regularizers import l2

model=Sequential([

Dense(256,activation='relu',input_shape=(1024,)),

BatchNormalization(),

Dropout(0.5),

Dense(64,activation='relu',kernel_regularizer=l2(0.001)),

Dropout(0.3),

Dense(8,activation='softmax')

])

model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

early=EarlyStopping(monitor='val_loss',patience=20,restore_best_weights=True)

history=model.fit(X_train_embed,y_train,
    validation_data=(X_test_embed,y_test),
    epochs=80,callbacks=[early]
)

Epoch 1/80
108/108 ━━━━━━━━━━━━━━━━━━━━ 8s 24ms/step - accuracy: 0.2622 - loss: 2.2469 - val_accuracy: 0.3160 - val_loss: 1.9975
Epoch 2/80
108/108 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.3348 - loss: 1.8544 - val_accuracy: 0.3704 - val_loss: 1.8133
Epoch 3/80
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.3802 - loss: 1.7271 - val_accuracy: 0.4421 - val_loss: 1.7109
Epoch 4/80
108/108 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.4172 - loss: 1.6404 - val_accuracy: 0.4583 - val_loss: 1.5603
Epoch 5/80
108/108 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.4253 - loss: 1.5788 - val_accuracy: 0.4491 - val_loss: 1.5483
Epoch 6/80
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.4378 - loss: 1.5420 - val_accuracy: 0.4676 - val_loss: 1.4851
Epoch 7/80
108/108 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.4627 - loss: 1.4709 - val_accuracy: 0.4699 - val_loss: 1.4620
Epoch 8/80
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.4679 - loss: 1.4665 - val_accura

In [ ]:
model.save("models/emotion_model.keras")

In [ ]:
from sklearn.metrics import confusion_matrix

y_pred_probs = model.predict(X_test_embed)

y_pred = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_test, y_pred)

labels = encoder.classes_

print(cm)

27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
[[22 10  4 18  0  1  3  0]
 [ 7 79  0  7  1  3  2  0]
 [ 9  3 45 13  2 21  5 13]
 [17 24  8 35  1 13 11  7]
 [ 1  0 17  8 66  8  7  7]
 [ 3  3 12 11  0 77  4  9]
 [ 3  9  3 16  2 10 79  6]
 [ 0  4 12 12  1  8  8 74]]
